# Temporal Walk-Forward Backtest

**Question:** *Is the model's profit edge stable across years, or did we just fit one vintage?*

Train on every loan issued **up to and including 2014**; test separately on **2015, 2016, 2017, 2018**. Report profit per 1,000 loans for each test year and the profit-retention ratio between the first and last test window.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import pandas as pd

from src import data_loader, backtest

df = data_loader.load_clean()
feature_cols = data_loader.feature_columns(df)
df['issue_year'].value_counts().sort_index()

In [ ]:
walk = backtest.walk_forward_backtest(df, feature_cols,
                                      train_until_year=2014)
walk

In [ ]:
backtest.profit_retention(walk)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(walk['test_year'].astype(str), walk['profit_per_1000'],
       color='#2980b9', label='Profit-optimized')
ax.bar(walk['test_year'].astype(str), walk['naive_profit_per_1000'],
       alpha=0.45, color='#e74c3c', label='Naive (approve all)')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Profit per 1,000 loans ($)')
ax.set_title('Walk-forward profit retention', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/walkforward.png', dpi=140, bbox_inches='tight')
plt.show()

**Interview answer:** the policy is stable if `profit_per_1000` doesn't collapse across test years. A retention ratio in the 0.7–1.2 range is healthy; below 0.5 is concerning and suggests concept drift.